# Brain Hemorrhage Detection with ResNeXt101

**Author:** Shaghayegh Shoushtari  
**Supervisor:** Dr. Mahmoud Farrokhian

This project detects intracranial hemorrhage in head CT slices using a ResNeXt101 convolutional neural network with an Efficient Channel Attention (ECA) module. The network keeps the original six hemorrhage-related outputs, while the main evaluation in this project focuses on the `any` output to determine whether hemorrhage is present.

## Before running

Install a compatible PyTorch/torchvision pair, NumPy, pandas, pydicom, OpenCV, scikit-learn and Matplotlib.

Provide the DICOM images, the train/validation/test CSV files and the trained ECA checkpoint. Each CSV should contain `Image` (or `ImageId`), `SeriesInstanceUID`, and the six target columns: `epidural`, `intraparenchymal`, `intraventricular`, `subarachnoid`, `subdural`, and `any`.

Update the dataset and checkpoint paths in the configuration section, then run the cells in order to prepare the images, load or train the model, and evaluate its performance.

## 1. Configuration

Set the paths for the DICOM images, dataset split files, trained checkpoint and output directory. The remaining parameters define the image size, batch size, learning rate, number of epochs and decision threshold used by the project.

In [ ]:
CONFIG = {'images_dir': 'data/dicom', 'train_csv': 'data/train.csv', 'valid_csv': 'data/valid.csv', 'test_csv': 'data/test.csv', 'checkpoint': 'weights/eca_resnext101.pt', 'backbone_weights': None, 'output_dir': 'outputs', 'patient_column': 'PatientID', 'device': 'auto', 'batch_size': 32, 'num_workers': 0, 'image_size': 256, 'epochs': 12, 'learning_rate': 1e-05, 'augment_training': True, 'on_dicom_error': 'raise', 'threshold_space': 'logit', 'threshold': 0.5, 'seed': 1, 'allow_legacy_pickle': False}


## 2. CT Image Preprocessing

Each DICOM image is converted to Hounsfield units. Three CT windows—brain, subdural and soft tissue—are then created from the same slice and stacked as three input channels. Finally, the image is resized to 256 × 256 pixels and normalized before being passed to the network.

In [ ]:
"""CT image windowing and normalization."""
import numpy as np
CHANNELS = ('brain', 'subdural', 'soft_tissue')
WINDOWS = ((40, 80), (80, 200), (40, 380))
MEAN = np.array([0.1738, 0.1433, 0.197], dtype=np.float32)
STD = np.array([0.3161, 0.285, 0.3111], dtype=np.float32)

def dicom_to_hu(dcm):
    pixels = np.asarray(dcm.pixel_array)
    if pixels.ndim != 2:
        raise ValueError('Expected one 2D CT slice, not a multiframe image.')
    pixels = pixels.astype(np.float32)
    slope = float(getattr(dcm, 'RescaleSlope', 1))
    intercept = float(getattr(dcm, 'RescaleIntercept', 0))
    if int(getattr(dcm, 'BitsStored', 16)) == 12 and int(getattr(dcm, 'PixelRepresentation', 1)) == 0 and (int(intercept) > -100):
        pixels = pixels + 1000
        pixels = np.where(pixels >= 4096, pixels - 4096, pixels)
        intercept = -1000
    hu = pixels * slope + intercept
    if not np.isfinite(hu).all():
        raise ValueError('Nonfinite CT intensity values.')
    return hu

def window_channels(hu):
    hu = np.asarray(hu, dtype=np.float32)
    if hu.ndim != 2 or not np.isfinite(hu).all():
        raise ValueError('Expected a finite 2D HU array.')
    channels = []
    for center, width in WINDOWS:
        lower = center - width / 2
        channels.append((np.clip(hu, lower, lower + width) - lower) / width)
    return np.stack(channels, axis=-1).astype(np.float32)

def normalize_channels(image):
    image = np.asarray(image, dtype=np.float32)
    if image.ndim != 3 or image.shape[-1] != 3:
        raise ValueError('Expected an H x W x 3 image.')
    return (image - MEAN) / STD

def prepare_image(dcm, size=256, augment=False):
    import cv2
    image = window_channels(dicom_to_hu(dcm))
    image = cv2.resize(image, (size, size), interpolation=cv2.INTER_LINEAR)
    if augment and np.random.random() < 0.5:
        image = image[:, ::-1]
    image = normalize_channels(image)
    return np.ascontiguousarray(image.transpose(2, 0, 1), dtype=np.float32)


## 3. Dataset Preparation

The training, validation and test CSV files contain the image identifiers and six hemorrhage labels. Training images can receive a small random augmentation, while validation and test images use fixed preprocessing. The dataset loader also keeps the image and series identifiers needed for evaluation.

In [ ]:
"""Read dataset labels and DICOM images."""
from pathlib import Path
import numpy as np
import pandas as pd
LABELS = ['epidural', 'intraparenchymal', 'intraventricular', 'subarachnoid', 'subdural', 'any']

def read_labels(csv_file):
    frame = pd.read_csv(csv_file)
    if 'Image' not in frame and 'ImageId' in frame:
        frame['Image'] = frame['ImageId']
    required = ['Image', 'SeriesInstanceUID'] + LABELS
    missing = [name for name in required if name not in frame]
    if missing:
        raise ValueError(f'{csv_file}: missing columns {missing}. Use the original wide split CSV, not the long RSNA label table.')
    if len(frame) == 0:
        raise ValueError(f'{csv_file}: empty partition.')
    if frame[['Image', 'SeriesInstanceUID']].isna().any().any():
        raise ValueError(f'{csv_file}: missing image or series identifiers.')
    frame['Image'] = frame['Image'].astype(str)
    frame['SeriesInstanceUID'] = frame['SeriesInstanceUID'].astype(str)
    if frame['Image'].duplicated().any():
        raise ValueError(f'{csv_file}: duplicate image identifiers.')
    values = frame[LABELS].to_numpy(dtype=np.float32)
    if not np.isfinite(values).all() or not np.isin(values, [0, 1]).all():
        raise ValueError(f'{csv_file}: all six labels must be 0 or 1.')
    if not np.array_equal(values[:, -1], values[:, :-1].max(axis=1)):
        raise ValueError(f'{csv_file}: Any disagrees with the five subtype labels.')
    return frame.reset_index(drop=True)

def audit_splits(frames, patient_column='PatientID'):
    """Reject overlapping images/series; check patients when IDs are provided."""
    names = list(frames)
    checked = ['Image', 'SeriesInstanceUID']
    patient_available = bool(patient_column) and all((patient_column in frame and frame[patient_column].notna().all() and frame[patient_column].astype(str).str.strip().ne('').all() for frame in frames.values()))
    if patient_available:
        checked.append(patient_column)
    for i, left in enumerate(names):
        for right in names[i + 1:]:
            for column in checked:
                common = set(frames[left][column].astype(str)) & set(frames[right][column].astype(str))
                if common:
                    raise ValueError(f'{column} overlaps between {left} and {right}: {len(common)} shared IDs. Fix the split before running.')
    return {'checked_identifiers': checked, 'patient_separation_checked': bool(patient_available), 'partition_sizes': {name: len(frame) for name, frame in frames.items()}, 'warning': None if patient_available else 'Patient separation could not be verified. Add patient identifiers to all split CSVs.'}

class IntracranialDataset:
    """A map-style dataset accepted by the PyTorch DataLoader."""

    def __init__(self, frame, images_dir, size=256, training=False, augment=False, on_error='raise'):
        if on_error not in ('raise', 'skip'):
            raise ValueError('on_error must be raise or skip.')
        self.frame = frame.reset_index(drop=True)
        self.images_dir = Path(images_dir)
        self.size = size
        self.augment = bool(training and augment)
        self.on_error = on_error

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        import pydicom
        row = self.frame.iloc[index]
        image_id = str(row['Image'])
        filename = image_id if image_id.lower().endswith('.dcm') else image_id + '.dcm'
        if Path(filename).name != filename:
            raise ValueError('Image IDs must be filenames, not directory paths.')
        path = self.images_dir / filename
        try:
            dcm = pydicom.dcmread(path)
            image = prepare_image(dcm, self.size, self.augment)
        except Exception as exc:
            message = f'{type(exc).__name__}: {exc}'
            if self.on_error == 'raise':
                raise RuntimeError(f'Cannot process {path}: {message}') from exc
            return {'error': message, 'image_id': image_id, 'path': str(path)}
        return {'image': image, 'labels': row[LABELS].to_numpy(dtype=np.float32), 'image_id': image_id}

def collate_slices(items):
    from torch.utils.data._utils.collate import default_collate
    rejected = [item for item in items if 'error' in item]
    valid = [item for item in items if 'error' not in item]
    return (default_collate(valid) if valid else None, rejected)


## 4. Logging

The logger records useful messages during training and evaluation and saves them in the project output directory.

In [ ]:
"""Create the project logger."""
import logging
from pathlib import Path

def setup_logger(name, save_dir, filename='run.log'):
    folder = Path(save_dir)
    folder.mkdir(parents=True, exist_ok=True)
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    logger.propagate = False
    for handler in list(logger.handlers):
        logger.removeHandler(handler)
        handler.close()
    formatter = logging.Formatter('%(asctime)s %(levelname)s %(message)s')
    for handler in (logging.StreamHandler(), logging.FileHandler(folder / filename, encoding='utf-8')):
        handler.setFormatter(formatter)
        logger.addHandler(handler)
    return logger


## 5. Model Checkpoints

The following functions save model checkpoints during training and load a trained checkpoint for later evaluation. The best checkpoint is selected according to validation loss.

In [ ]:
"""Save and load model checkpoints."""
from pathlib import Path
import math
import os

def read_checkpoint(path, device='cpu', allow_legacy_pickle=False):
    import torch
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f'Checkpoint not found: {path}')
    try:
        return torch.load(path, map_location=device, weights_only=True)
    except Exception as exc:
        if not allow_legacy_pickle:
            raise RuntimeError('Cannot safely read this checkpoint. Original checkpoints may store a loss object. Only for your own trusted file, set allow_legacy_pickle=true in config.json.') from exc
        return torch.load(path, map_location=device, weights_only=False)

def state_dict_from_checkpoint(checkpoint):
    if not isinstance(checkpoint, dict):
        raise ValueError('Expected a state dictionary or a checkpoint dictionary.')
    state = checkpoint.get('model_state_dict', checkpoint.get('state_dict', checkpoint))
    if not isinstance(state, dict) or not state:
        raise ValueError('Checkpoint has no model state dictionary.')
    if all((key.startswith('module.') for key in state)):
        state = {key[len('module.'):]: value for key, value in state.items()}
    return state

def load_model_weights(model, path, device='cpu', allow_legacy_pickle=False):
    checkpoint = read_checkpoint(path, device, allow_legacy_pickle)
    model.load_state_dict(state_dict_from_checkpoint(checkpoint), strict=True)
    return checkpoint

def save_checkpoint(path, model, optimizer, epoch, validation_loss, metadata):
    import torch
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    torch.save({'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict(), 'epoch': int(epoch), 'validation_loss': float(validation_loss), 'metadata': metadata}, temporary)
    os.replace(temporary, path)

class SaveBestModel:

    def __init__(self, path):
        self.path = Path(path)
        self.best_valid_loss = float('inf')

    def __call__(self, loss, model, optimizer, epoch, metadata):
        if math.isfinite(loss) and loss < self.best_valid_loss:
            self.best_valid_loss = float(loss)
            save_checkpoint(self.path, model, optimizer, epoch, loss, metadata)
            return True
        return False


## 6. ResNeXt101 with Efficient Channel Attention

ResNeXt101 is used as the convolutional backbone for feature extraction. The ECA module gives additional attention to informative feature channels. After global average pooling, the final linear layer produces six raw outputs corresponding to the five hemorrhage subtypes and the `any` label.

In [ ]:
"""ResNeXt101 model with Efficient Channel Attention."""
import torch
from torch import nn

class eca_layer(nn.Module):

    def __init__(self, channel=2048, k_size=3):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, kernel_size=k_size, padding=(k_size - 1) // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        weights = self.avg_pool(x)
        weights = self.conv(weights.squeeze(-1).transpose(-1, -2))
        weights = self.sigmoid(weights.transpose(-1, -2).unsqueeze(-1))
        return x * weights.expand_as(x)

class ECA_ResNeXtModel(nn.Module):

    def __init__(self, backbone=None):
        super().__init__()
        if backbone is None:
            from torchvision.models import resnext101_32x8d
            backbone = resnext101_32x8d(weights=None)
        self.base = nn.Sequential(*list(backbone.children())[:-2])
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.eca = eca_layer(2048, k_size=3)
        self.relu = nn.ReLU(inplace=True)
        self.fc = nn.Sequential(nn.Linear(2048, 6))

    def forward(self, image):
        features = self.base(image)
        features = features + self.eca(features)
        pooled = self.avgpool(features).reshape(-1, 2048)
        return (self.fc(pooled), pooled)

def build_model(backbone_weights=None, allow_legacy_pickle=False):
    if backbone_weights:
        from torchvision.models import resnext101_32x8d
        backbone = resnext101_32x8d(weights=None)
        checkpoint = read_checkpoint(backbone_weights, 'cpu', allow_legacy_pickle)
        backbone.load_state_dict(state_dict_from_checkpoint(checkpoint), strict=True)
        return ECA_ResNeXtModel(backbone)
    return ECA_ResNeXtModel()


## 7. Model Evaluation

For the final binary evaluation, the project uses the `any` output. The raw model score is converted to a probability with the sigmoid function, and a decision threshold is applied. Accuracy, precision, sensitivity, specificity, F1 score, AUC and the confusion matrix are then calculated from the test predictions.

In [ ]:
"""Evaluate the Any output and calculate classification metrics."""
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score

def sigmoid(logits):
    logits = np.asarray(logits, dtype=np.float64)
    result = np.empty_like(logits)
    positive = logits >= 0
    result[positive] = 1 / (1 + np.exp(-logits[positive]))
    exponent = np.exp(logits[~positive])
    result[~positive] = exponent / (1 + exponent)
    return result

def any_decisions(logits, threshold=0.5, threshold_space='logit'):
    logits = np.asarray(logits, dtype=np.float64)
    if not np.isfinite(logits).all():
        raise ValueError('Nonfinite predictions.')
    if threshold_space == 'logit':
        return (logits >= threshold).astype(np.int64)
    if threshold_space == 'probability':
        if not 0 <= threshold <= 1:
            raise ValueError('A probability threshold must be between 0 and 1.')
        return (sigmoid(logits) >= threshold).astype(np.int64)
    raise ValueError('threshold_space must be logit or probability.')

def calculate_metrics(truth, logits, threshold=0.5, threshold_space='logit'):
    truth = np.asarray(truth).reshape(-1)
    logits = np.asarray(logits, dtype=np.float64).reshape(-1)
    if len(truth) == 0 or len(truth) != len(logits) or (not np.isin(truth, [0, 1]).all()):
        raise ValueError('Expected nonempty binary labels and matching logits.')
    predicted = any_decisions(logits, threshold, threshold_space)
    tn, fp, fn, tp = confusion_matrix(truth, predicted, labels=[0, 1]).ravel()
    return {'accuracy': float(accuracy_score(truth, predicted)), 'precision': float(precision_score(truth, predicted, zero_division=0)), 'sensitivity': float(recall_score(truth, predicted, zero_division=0)), 'specificity': float(tn / (tn + fp)) if tn + fp else None, 'f1': float(f1_score(truth, predicted, zero_division=0)), 'auc': float(roc_auc_score(truth, logits)) if len(np.unique(truth)) == 2 else None, 'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp), 'evaluated_slices': len(truth), 'positive_slices': int(truth.sum()), 'threshold': float(threshold), 'threshold_space': threshold_space, 'equivalent_probability_threshold': float(sigmoid([threshold])[0]) if threshold_space == 'logit' else float(threshold)}

def save_plots(truth, logits, metrics, folder):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    from sklearn.metrics import roc_curve
    folder = __import__('pathlib').Path(folder)
    matrix = np.array([[metrics['tn'], metrics['fp']], [metrics['fn'], metrics['tp']]])
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.imshow(matrix, cmap='Blues')
    for (row, column), count in np.ndenumerate(matrix):
        ax.text(column, row, str(count), ha='center', va='center')
    ax.set(xticks=[0, 1], yticks=[0, 1], xticklabels=['Absent', 'Present'], yticklabels=['Absent', 'Present'], xlabel='Predicted Any', ylabel='True Any', title='Slice-level confusion matrix')
    fig.tight_layout()
    fig.savefig(folder / 'confusion_matrix.png', dpi=160)
    plt.close(fig)
    if metrics['auc'] is not None:
        fpr, tpr, _ = roc_curve(truth, logits)
        fig, ax = plt.subplots(figsize=(5, 4))
        ax.plot(fpr, tpr, label=f"AUC = {metrics['auc']:.4f}")
        ax.plot([0, 1], [0, 1], '--', color='gray')
        ax.set(xlabel='False positive rate', ylabel='True positive rate', title='Any ROC curve')
        ax.legend()
        fig.tight_layout()
        fig.savefig(folder / 'roc_curve.png', dpi=160)
        plt.close(fig)


## 8. Training and Testing

During training, all six labels are optimized using `BCEWithLogitsLoss` and the Adam optimizer. Validation loss is calculated after each epoch so that the best model can be saved. The evaluation function loads the selected model and calculates the final metrics for the `any` output on the test set.

In [ ]:
"""Training and evaluation functions for the ECA-ResNeXt101 model."""
import argparse
import json
import random
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd

def read_config(path):
    if isinstance(path, dict):
        config = dict(path)
        base_dir = Path.cwd()
    else:
        path = Path(path).resolve()
        base_dir = path.parent
        config = json.loads(path.read_text(encoding='utf-8'))
    for key in ['images_dir', 'train_csv', 'valid_csv', 'test_csv', 'checkpoint', 'backbone_weights', 'output_dir']:
        if config.get(key):
            value = Path(config[key]).expanduser()
            config[key] = str(value if value.is_absolute() else base_dir / value)
    if config['image_size'] != 256:
        raise ValueError('This report uses 256x256 images. Keep image_size=256.')
    for key in ['batch_size', 'epochs']:
        if not isinstance(config[key], int) or config[key] < 1:
            raise ValueError(f'{key} must be a positive integer.')
    if not isinstance(config['num_workers'], int) or config['num_workers'] < 0:
        raise ValueError('num_workers must be a nonnegative integer.')
    if not np.isfinite(config['learning_rate']) or config['learning_rate'] <= 0:
        raise ValueError('learning_rate must be positive and finite.')
    if not np.isfinite(config['threshold']):
        raise ValueError('threshold must be finite.')
    any_decisions([0], config['threshold'], config['threshold_space'])
    if config['on_dicom_error'] not in ['raise', 'skip']:
        raise ValueError('on_dicom_error must be raise or skip.')
    return config

def check_data(config):
    frames = {name: read_labels(config[name + '_csv']) for name in ['train', 'valid', 'test']}
    audit = audit_splits(frames, config.get('patient_column'))
    image_dir = Path(config['images_dir'])
    if not image_dir.is_dir():
        raise FileNotFoundError(f'Image directory not found: {image_dir}')
    missing = []
    for name, frame in frames.items():
        for image_id in frame['Image']:
            filename = image_id if image_id.lower().endswith('.dcm') else image_id + '.dcm'
            if Path(filename).name != filename:
                raise ValueError('Image IDs must be filenames, not directory paths.')
            if not (image_dir / filename).is_file():
                missing.append({'split': name, 'image_id': image_id})
    if missing:
        raise FileNotFoundError(f'{len(missing)} DICOM files are missing. First missing image: {missing[0]}. Fix the paths before running.')
    return (frames, audit)

def runtime_device(config):
    import torch
    device = config['device']
    if device == 'auto':
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
    if str(device).startswith('cuda') and (not torch.cuda.is_available()):
        raise RuntimeError('CUDA was requested but is not available.')
    return torch.device(device)

def make_loader(frame, config, training=False):
    from torch.utils.data import DataLoader
    dataset = IntracranialDataset(frame, config['images_dir'], config['image_size'], training, config['augment_training'], config['on_dicom_error'])
    return DataLoader(dataset, batch_size=config['batch_size'], shuffle=training, num_workers=config['num_workers'], collate_fn=collate_slices)

def model_metadata(config):
    return {'model': 'ECA_ResNeXtModel', 'backbone': 'resnext101_32x8d', 'labels': LABELS, 'image_size': config['image_size'], 'channels': list(CHANNELS), 'mean': MEAN.tolist(), 'std': STD.tolist(), 'training_augmentation': 'horizontal_flip' if config['augment_training'] else 'none'}

def check_checkpoint_metadata(checkpoint, config):
    if 'metadata' not in checkpoint:
        return
    expected = model_metadata(config)
    for key in ['model', 'labels', 'image_size', 'channels', 'mean', 'std']:
        if key in checkpoint['metadata'] and checkpoint['metadata'][key] != expected[key]:
            raise ValueError(f'Checkpoint metadata mismatch for {key}. Use matching preprocessing and weights.')

def run_epoch(model, loader, criterion, device, log, rejected_file, optimizer=None):
    import torch
    training = optimizer is not None
    model.train(training)
    total_loss, seen, skipped = (0.0, 0, 0)
    for batch, rejected in loader:
        for item in rejected:
            rejected_file.write(json.dumps(item) + '\n')
            rejected_file.flush()
            log.warning('Rejected image %s: %s', item['image_id'], item['error'])
        skipped += len(rejected)
        if batch is None:
            continue
        images = batch['image'].to(device, dtype=torch.float32)
        labels = batch['labels'].to(device, dtype=torch.float32)
        with torch.set_grad_enabled(training):
            logits, _ = model(images)
            loss = criterion(logits, labels)
            if not torch.isfinite(loss):
                raise RuntimeError('Nonfinite loss. Training/evaluation stopped.')
            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * len(images)
        seen += len(images)
    if seen == 0:
        raise RuntimeError('No readable images were processed.')
    return (total_loss / seen, seen, skipped)

def train(config, frames, output, log):
    import torch
    checkpoint_path = config.get('checkpoint')
    if not (checkpoint_path and Path(checkpoint_path).is_file()) and (not config.get('backbone_weights')):
        raise FileNotFoundError('Transfer learning requires an existing ECA checkpoint or local WSL backbone_weights. No random-weight training or automatic download is performed.')
    device = runtime_device(config)
    model = build_model(config.get('backbone_weights'), config['allow_legacy_pickle'])
    if checkpoint_path and Path(checkpoint_path).is_file():
        checkpoint = load_model_weights(model, checkpoint_path, 'cpu', config['allow_legacy_pickle'])
        check_checkpoint_metadata(checkpoint, config)
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config['learning_rate'])
    criterion = torch.nn.BCEWithLogitsLoss()
    training_loader = make_loader(frames['train'], config, training=True)
    valid_loader = make_loader(frames['valid'], config, training=False)
    best = SaveBestModel(output / 'best_model.pt')
    metadata = model_metadata(config)
    history = []
    with (output / 'rejected_training_images.jsonl').open('w', encoding='utf-8') as rejected:
        for epoch in range(1, config['epochs'] + 1):
            tr_loss, tr_n, tr_skip = run_epoch(model, training_loader, criterion, device, log, rejected, optimizer)
            val_loss, val_n, val_skip = run_epoch(model, valid_loader, criterion, device, log, rejected)
            improved = best(val_loss, model, optimizer, epoch, metadata)
            row = {'epoch': epoch, 'training_loss': tr_loss, 'validation_loss': val_loss, 'training_slices': tr_n, 'validation_slices': val_n, 'training_skipped': tr_skip, 'validation_skipped': val_skip, 'best_saved': improved}
            history.append(row)
            pd.DataFrame(history).to_csv(output / 'training_history.csv', index=False)
            log.info('Epoch %d train=%.6f validation=%.6f best_saved=%s', epoch, tr_loss, val_loss, improved)
            save_checkpoint(output / 'last_model.pt', model, optimizer, epoch, val_loss, metadata)
    log.info('Training finished. Configure checkpoint to %s before evaluating.', output / 'best_model.pt')

def evaluate(config, frames, output, log):
    import torch
    device = runtime_device(config)
    if not config.get('checkpoint'):
        raise ValueError('Set checkpoint to the trained ECA model file.')
    model = build_model()
    checkpoint = load_model_weights(model, config['checkpoint'], 'cpu', config['allow_legacy_pickle'])
    check_checkpoint_metadata(checkpoint, config)
    model.to(device).eval()
    criterion = torch.nn.BCEWithLogitsLoss()
    truth, raw_scores, rows = ([], [], [])
    loss_sum, seen, skipped = (0.0, 0, 0)
    loader = make_loader(frames['test'], config, training=False)
    with (output / 'rejected_test_images.jsonl').open('w', encoding='utf-8') as rejected_file:
        with torch.no_grad():
            for batch, rejected in loader:
                skipped += len(rejected)
                for item in rejected:
                    rejected_file.write(json.dumps(item) + '\n')
                    rejected_file.flush()
                    log.warning('Rejected image %s: %s', item['image_id'], item['error'])
                if batch is None:
                    continue
                images = batch['image'].to(device, dtype=torch.float32)
                labels = batch['labels'].to(device, dtype=torch.float32)
                logits, _ = model(images)
                loss = criterion(logits, labels)
                if not torch.isfinite(loss):
                    raise RuntimeError('Nonfinite test loss.')
                any_logits = logits[:, 5].cpu().numpy().astype(np.float64)
                labels_any = labels[:, 5].cpu().numpy().astype(np.int64)
                probabilities = sigmoid(any_logits)
                predicted = any_decisions(any_logits, config['threshold'], config['threshold_space'])
                for image_id, label, score, probability, prediction in zip(batch['image_id'], labels_any, any_logits, probabilities, predicted):
                    rows.append({'image_id': image_id, 'true_any': int(label), 'logit_any': float(score), 'probability_any': float(probability), 'predicted_any': int(prediction)})
                truth.extend(labels_any)
                raw_scores.extend(any_logits)
                loss_sum += loss.item() * len(images)
                seen += len(images)
    metrics = calculate_metrics(truth, raw_scores, config['threshold'], config['threshold_space'])
    metrics.update({'six_output_bce_loss': loss_sum / seen, 'requested_test_slices': len(frames['test']), 'skipped_slices': skipped, 'checkpoint': config['checkpoint'], 'source': 'new evaluation, not copied historical results'})
    pd.DataFrame(rows).to_csv(output / 'any_predictions.csv', index=False)
    (output / 'any_metrics.json').write_text(json.dumps(metrics, indent=2, allow_nan=False), encoding='utf-8')
    save_plots(np.asarray(truth), np.asarray(raw_scores), metrics, output)
    log.info('Any evaluation: %s', metrics)
    log.info('Compare results with the report only after confirming checkpoint, split and preprocessing match.')
    return metrics

def run(mode='evaluate', config_path=None):
    config = read_config(CONFIG if config_path is None else config_path)
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    output = Path(config['output_dir']) / (mode + '_' + stamp)
    output.mkdir(parents=True, exist_ok=False)
    log = setup_logger('ich_project', output)
    frames, audit = check_data(config)
    (output / 'split_audit.json').write_text(json.dumps(audit, indent=2), encoding='utf-8')
    (output / 'run_config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
    log.info('Split audit: %s', audit)
    if mode == 'check-data':
        log.info('CSV structure, image existence and available identifier overlap checks passed. DICOM decoding is checked during execution.')
        return audit
    import torch
    random.seed(config['seed'])
    np.random.seed(config['seed'])
    torch.manual_seed(config['seed'])
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(config['seed'])
    if mode == 'train':
        return train(config, frames, output, log)
    if mode == 'evaluate':
        return evaluate(config, frames, output, log)
    raise ValueError('Mode must be check-data, train or evaluate.')


## 9. Running the Project

After updating the paths in the configuration section, choose the required action below.

- `RUN_DATA_CHECK` verifies the dataset files and split information.
- `RUN_TRAINING` trains the ECA-ResNeXt101 model and saves checkpoints.
- `RUN_EVALUATION` loads a trained checkpoint and evaluates the `any` output on the test set.

Enable only one action at a time. Evaluation outputs, including predictions and calculated metrics, are saved in the output directory.

In [ ]:
RUN_DATA_CHECK = False
RUN_EVALUATION = False
RUN_TRAINING = False

actions = [('check-data', RUN_DATA_CHECK), ('evaluate', RUN_EVALUATION), ('train', RUN_TRAINING)]
if sum(bool(enabled) for _, enabled in actions) > 1:
    raise ValueError('Enable one action at a time.')
for mode, enabled in actions:
    if enabled:
        result = run(mode, CONFIG)
        if result is not None:
            print(result)


## 10. Project Notes

The notebook combines the data preparation, model definition, training and evaluation steps in a single workflow. The model retains six outputs for compatibility with the trained architecture, while this Bachelor’s project focuses on the binary `any` label for hemorrhage detection.

For reproducible evaluation, use the same dataset split, preprocessing settings and trained checkpoint that belong to the experiment being evaluated.